# Lab 10: Evaluation, Ablations, and Building Your Evidence Base

**Before you start: go to Runtime → Change runtime type and select GPU.**

By week 10 you should be approaching a final model. The question is no 
longer *can I get it to work?* but *can I prove that my approach is good?*

A result without evidence is just a number. A result with proper evaluation, 
ablations, and honest comparison to baselines is a contribution.

**This lab covers:**
1. Train/val/test discipline — the rules and the common mistakes
2. Choosing and reporting the right metric for your task
3. Structuring ablations that actually prove something
4. Comparing fairly against baselines and prior work

**Session structure:**
- First 30 minutes: work through this notebook
- Mid-session: **mid-semester showcase** — 3-minute demos from each group
- Remaining time: apply the evaluation framework to your own project

**Goal for today:** leave with a complete draft results table 
and a clear plan for the ablations in your final report.

## Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, average_precision_score,
    confusion_matrix, ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt
import numpy as np
import zipfile, gdown, json, random
from pathlib import Path
from copy import deepcopy

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# Load dataset
gdown.download('https://drive.google.com/uc?id=1KDMC39ba1MAL83FLLoSVSJY2KOmFR1aj', 'data.zip', quiet=True)
with zipfile.ZipFile('data.zip', 'r') as z:
    z.extractall('.')
data_dir = Path('data')

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_ds = datasets.ImageFolder(data_dir / 'train', transform=train_transform)
val_ds   = datasets.ImageFolder(data_dir / 'val',   transform=val_transform)
train_dl = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2)
val_dl   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2)

CLASS_NAMES = train_ds.classes
NUM_CLASSES = len(CLASS_NAMES)
print(f'Classes: {CLASS_NAMES}')

## 1. Train / Val / Test Discipline

This seems obvious but is violated surprisingly often, sometimes in 
subtle ways. The rules are simple; the reasons behind them matter.

### The three-split protocol

| Split | Used for | How many times? |
|---|---|---|
| **Training** | Gradient updates | Many times |
| **Validation** | Hyperparameter tuning, model selection | Many times |
| **Test** | Final reported result | **Exactly once** |

The test set must never be used during development. The moment you 
use test set performance to make any decision — even a small one like 
choosing between two augmentation strategies — it becomes a second 
validation set and your final number is optimistically biased.

### Common violations

**Data leakage:** test images used during training, directly or indirectly. 
Classic example: normalising the entire dataset (train + test together) 
before splitting — the normalisation statistics carry test information 
into training.

**Multiple testing:** selecting the model with the best test performance 
out of many runs. Even with a fixed test set, running 20 experiments and 
reporting the best inflates the number.

**Validation set used as test set:** the most common student mistake. 
If you tuned on it, it is a validation set, not a test set — even if 
you called it a test set.

In [ ]:
def create_proper_splits(dataset, train_frac=0.7, val_frac=0.15, seed=42):
    """
    Create stratified train/val/test splits from a single dataset.
    Stratified means each split has the same class proportions.

    Use this when you only have one dataset and need to create splits yourself.
    If your dataset already has a predefined test split, use that.
    """
    random.seed(seed)
    assert abs(train_frac + val_frac - 0.85) < 1e-9 or train_frac + val_frac < 1.0

    # Group indices by class
    class_indices = {}
    for idx, (_, label) in enumerate(dataset.samples):
        class_indices.setdefault(label, []).append(idx)

    train_idx, val_idx, test_idx = [], [], []

    for label, indices in class_indices.items():
        random.shuffle(indices)
        n       = len(indices)
        n_train = int(n * train_frac)
        n_val   = int(n * val_frac)
        train_idx.extend(indices[:n_train])
        val_idx.extend(indices[n_train:n_train + n_val])
        test_idx.extend(indices[n_train + n_val:])

    print(f'Stratified split:')
    print(f'  Train: {len(train_idx)} ({len(train_idx)/len(dataset)*100:.1f}%)')
    print(f'  Val:   {len(val_idx)} ({len(val_idx)/len(dataset)*100:.1f}%)')
    print(f'  Test:  {len(test_idx)} ({len(test_idx)/len(dataset)*100:.1f}%)')

    # Verify class balance across splits
    from collections import Counter
    for name, idx_list in [('Train', train_idx), ('Val', val_idx), ('Test', test_idx)]:
        counts = Counter(dataset.targets[i] for i in idx_list)
        fracs  = {dataset.classes[k]: f'{v/len(idx_list)*100:.1f}%'
                  for k, v in sorted(counts.items())}
        print(f'  {name} class fractions: {fracs}')

    return train_idx, val_idx, test_idx


# Demonstrate: combine our train and val datasets, then re-split properly
from torch.utils.data import ConcatDataset

# We use a version without transforms for splitting purposes
full_ds = datasets.ImageFolder(data_dir / 'train', transform=val_transform)
train_idx, val_idx, test_idx = create_proper_splits(full_ds)

## 2. Choosing and Reporting Metrics

The right metric depends on your task and what errors matter most. 
A single number rarely tells the whole story — your report should 
include at least two metrics and explain why you chose them.

### Metric selection guide

In [ ]:
# Train a model to evaluate
def train_model(epochs=10, lr=1e-3, seed=42):
    torch.manual_seed(seed)
    m    = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)
    m    = m.to(device)
    opt  = torch.optim.Adam(m.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        m.train()
        for imgs, lbls in train_dl:
            imgs, lbls = imgs.to(device), lbls.to(device)
            opt.zero_grad()
            loss = crit(m(imgs), lbls)
            loss.backward()
            opt.step()
    return m

print('Training model...')
model = train_model(epochs=10)

# Collect predictions
model.eval()
all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for imgs, lbls in val_dl:
        logits = model(imgs.to(device))
        all_probs.extend(F.softmax(logits, dim=1).cpu().numpy())
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_labels.extend(lbls.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs  = np.array(all_probs)

print(f'Collected {len(all_preds)} predictions.')

In [ ]:
def compute_all_metrics(labels, preds, probs, class_names):
    """
    Compute a comprehensive set of classification metrics.
    Use this as a starting point and select the metrics relevant to your task.
    """
    n_classes = len(class_names)
    metrics   = {}

    # --- Basic accuracy ---
    metrics['accuracy'] = accuracy_score(labels, preds)

    # --- Per-class and macro F1 ---
    metrics['macro_f1']  = f1_score(labels, preds, average='macro')
    metrics['micro_f1']  = f1_score(labels, preds, average='micro')
    per_class_f1         = f1_score(labels, preds, average=None)

    # --- Precision and recall ---
    metrics['macro_precision'] = precision_score(labels, preds, average='macro')
    metrics['macro_recall']    = recall_score(labels, preds,    average='macro')

    # --- AUC-ROC ---
    try:
        metrics['auc_roc'] = roc_auc_score(
            labels, probs, multi_class='ovr', average='macro'
        )
    except Exception:
        metrics['auc_roc'] = float('nan')

    # --- Per-class accuracy ---
    cm              = confusion_matrix(labels, preds)
    per_class_acc   = cm.diagonal() / cm.sum(axis=1)

    # Print summary
    print('=== Metric Summary ===')
    print(f'  Accuracy:         {metrics["accuracy"]:.4f}')
    print(f'  Macro F1:         {metrics["macro_f1"]:.4f}')
    print(f'  Macro Precision:  {metrics["macro_precision"]:.4f}')
    print(f'  Macro Recall:     {metrics["macro_recall"]:.4f}')
    print(f'  AUC-ROC (OvR):    {metrics["auc_roc"]:.4f}')

    print(f'\n  Per-class accuracy:')
    for name, acc, f1 in zip(class_names, per_class_acc, per_class_f1):
        print(f'    {name:<20} acc={acc:.3f}  F1={f1:.3f}')

    return metrics, per_class_acc, per_class_f1


metrics, per_class_acc, per_class_f1 = compute_all_metrics(
    all_labels, all_preds, all_probs, CLASS_NAMES
)

print()
print('Metric selection guide:')
print('  Balanced classes, simple task    → Accuracy')
print('  Imbalanced classes               → Macro F1')
print('  Asymmetric error costs           → Precision + Recall separately')
print('  Binary classification            → AUC-ROC')
print('  Multi-label classification       → Mean Average Precision (mAP)')
print('  Object detection                 → mAP@IoU threshold')
print('  Segmentation                     → Mean IoU')
print('  Generation                       → FID + human evaluation')

## 3. Structuring Ablations

An ablation study removes or changes one component at a time to measure 
its contribution. Good ablations answer the question: *does this specific 
design choice actually help?*

### What makes a good ablation

**One change at a time.** If you change two things simultaneously, 
you cannot attribute the result to either one.

**Same training protocol.** All ablation conditions must use the 
same number of epochs, learning rate, and random seed — or report 
averages over multiple seeds.

**Report the full table.** Do not cherry-pick conditions. If you tried 
10 ablations and 3 helped, report all 10.

**Include the full model as baseline.** The ablation table should 
always have a row for 'Full model' that all other rows are compared against.

In [ ]:
def run_ablation(name, model_fn, train_dl, val_dl,
                 epochs=10, lr=1e-3, seed=42):
    """
    Run one ablation condition and return its metrics.
    """
    torch.manual_seed(seed)
    model = model_fn().to(device)
    opt   = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()), lr=lr
    )
    crit  = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        for imgs, lbls in train_dl:
            imgs, lbls = imgs.to(device), lbls.to(device)
            opt.zero_grad()
            loss = crit(model(imgs), lbls)
            loss.backward()
            opt.step()

    model.eval()
    preds, labels, probs = [], [], []
    with torch.no_grad():
        for imgs, lbls in val_dl:
            logits = model(imgs.to(device))
            probs.extend(F.softmax(logits, dim=1).cpu().numpy())
            preds.extend(logits.argmax(1).cpu().numpy())
            labels.extend(lbls.numpy())

    preds, labels, probs = np.array(preds), np.array(labels), np.array(probs)
    acc = accuracy_score(labels, preds)
    f1  = f1_score(labels, preds, average='macro')
    print(f'  {name:<35}  acc={acc:.3f}  macro_f1={f1:.3f}')
    return {'name': name, 'acc': acc, 'macro_f1': f1}


# Define ablation conditions
# Each changes exactly one thing from the full model

def full_model():
    """Full model — pretrained ResNet-18, fine-tuned with augmentation."""
    m    = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)
    return m

def no_pretrain():
    """No pretraining — random initialisation."""
    m    = models.resnet18(weights=None)
    m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)
    return m

def frozen_backbone():
    """Feature extraction only — backbone frozen."""
    m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    for name, p in m.named_parameters():
        if 'fc' not in name:
            p.requires_grad = False
    m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)
    return m

def smaller_backbone():
    """Smaller backbone — ResNet-18 vs ResNet-50 (we use a tiny CNN here)."""
    return nn.Sequential(
        nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d((4,4)),
        nn.Flatten(),
        nn.Linear(64*4*4, NUM_CLASSES)
    )

# No-augmentation training loader
no_aug_ds = datasets.ImageFolder(data_dir / 'train', transform=val_transform)
no_aug_dl = DataLoader(no_aug_ds, batch_size=32, shuffle=True, num_workers=2)

print('Running ablation study (this will take several minutes)...')
ablation_results = []

# Full model (baseline for ablation)
ablation_results.append(run_ablation('Full model (baseline)', full_model, train_dl, val_dl))

# Remove pretraining
ablation_results.append(run_ablation('No pretraining', no_pretrain, train_dl, val_dl))

# Freeze backbone
ablation_results.append(run_ablation('Frozen backbone', frozen_backbone, train_dl, val_dl))

# No augmentation
ablation_results.append(run_ablation('No data augmentation', full_model, no_aug_dl, val_dl))

# Smaller model
ablation_results.append(run_ablation('Small CNN (no pretrain)', smaller_backbone, train_dl, val_dl))

In [ ]:
# Print a publication-style ablation table
baseline_acc = ablation_results[0]['acc']
baseline_f1  = ablation_results[0]['macro_f1']

print('Ablation Study Results')
print('=' * 72)
print(f"  {'Model / Condition':<35} {'Acc':>7} {'Δ Acc':>8} {'Macro F1':>9}")
print('  ' + '-' * 61)

for r in ablation_results:
    delta = r['acc'] - baseline_acc
    sign  = '+' if delta >= 0 else ''
    marker = ' ←' if r['name'] == ablation_results[0]['name'] else ''
    print(f"  {r['name']:<35} {r['acc']:>7.3f} {sign+f'{delta:.3f}':>8} "
          f"{r['macro_f1']:>9.3f}{marker}")

print('=' * 72)
print('  ← = Full model (baseline for comparison)')
print()

# Visualise the ablation results
names  = [r['name'] for r in ablation_results]
accs   = [r['acc']  for r in ablation_results]
deltas = [r['acc'] - baseline_acc for r in ablation_results]

fig, ax = plt.subplots(figsize=(10, 5))
colors  = ['steelblue' if d >= 0 else 'tomato' for d in deltas]
bars    = ax.barh(names[::-1], [d * 100 for d in deltas[::-1]],
                  color=colors[::-1])
ax.axvline(0, color='black', linewidth=1)
ax.set_xlabel('Δ Accuracy vs full model (%)')
ax.set_title('Ablation study — contribution of each component')
for bar, delta in zip(bars, [d * 100 for d in deltas[::-1]]):
    x = bar.get_width()
    ax.text(x + (0.3 if x >= 0 else -0.3), bar.get_y() + bar.get_height()/2,
            f'{x:+.1f}%', va='center', ha='left' if x >= 0 else 'right', fontsize=9)
plt.tight_layout()
plt.show()

print('Reading the table:')
print('  Negative delta = removing this component hurts performance')
print('  Near-zero delta = this component has little effect')
print('  Positive delta = this component actually hurts (consider removing it)')

## 4. Comparing Against Baselines and Prior Work

Your results only mean something in context. Every results table 
needs at least one reference point to compare against:

- **Trivial baseline** (random chance, majority class) — the floor
- **Simple baseline** (your week 6 baseline) — the starting point
- **Your best model** — the contribution
- **Prior work** (if available) — how you compare to the state of the art

### Finding prior work

Search [Papers with Code](https://paperswithcode.com/) for your dataset. 
If your dataset is custom, find the most similar published dataset and 
cite its benchmark results as context.

If no prior work exists: that is worth mentioning. 
"To our knowledge, no prior work has benchmarked on this dataset" 
is a legitimate statement — it makes your baseline more valuable, 
not less.

In [ ]:
# Template for a complete results table
# Adapt this format for your report

# Trivial baselines
from collections import Counter
label_counts = Counter(train_ds.targets)
n_total      = sum(label_counts.values())
random_acc   = 1 / NUM_CLASSES
majority_acc = max(label_counts.values()) / n_total

print('Complete Results Table')
print('Dataset: Cats / Dogs / Horses (300 train per class)')
print('Metric: Accuracy (balanced classes — appropriate here)')
print()
print(f"{'Model':<35} {'Val Acc':>8} {'Notes'}")
print('-' * 70)
print(f"{'Random chance':<35} {random_acc:>8.3f} {'lower bound'}")
print(f"{'Majority class':<35} {majority_acc:>8.3f} {'lower bound'}")
print(f"{'Small CNN (no pretrain)':<35} {ablation_results[-1]['acc']:>8.3f} {'baseline'}")
for r in ablation_results[:-1]:
    tag = 'full model (ours)' if r['name'] == ablation_results[0]['name'] else 'ablation'
    print(f"{r['name']:<35} {r['acc']:>8.3f} {tag}")
print('-' * 70)
print()
print('Tips for your results table:')
print('  - Always include random chance as the first row')
print('  - Report the same metric consistently across all rows')
print('  - Mark your main result clearly (bold in the paper)')
print('  - Note if a comparison used different train/val splits')
print('  - Report val accuracy during development, test accuracy only once at the end')

## Mid-Semester Showcase

**Each group now has 3 minutes to show where their project is.**

No slides needed — just show your current result. This could be:
- Your best result so far and the loss curves
- A confusion matrix or worst-case examples
- A generated image or detection output
- A problem you are stuck on and what you have tried

The purpose is not to impress — it is to make your project real and 
to learn from what other groups are doing.

**Format:** Groups present in sequence. 3 minutes each, no questions 
during the presentation. Brief instructor comment after each group.

---

## Applying this to your project

After the showcase, use the remaining session time to build 
your evidence base.

**Step 1 — Audit your splits.**
Are your train/val/test splits properly separated? 
Have you ever used test set performance to make a decision?
If yes: relabel that split as a second validation set and 
hold out a fresh test split.

**Step 2 — Choose your primary metric.**
Run `compute_all_metrics` and decide which metric is the 
right primary metric for your task. Be ready to justify the choice.

**Step 3 — Design your ablation table.**
List the design choices in your model that you can ablate:
- Pretrained vs random init
- Augmentation on/off
- Backbone size (ResNet-18 vs ResNet-50)
- Dropout rate
- Learning rate schedule
- Any custom components in your architecture

You do not need to run all of them today — but you need the table 
to exist in your final report, so plan which ablations you will run.

**Step 4 — Draft your results table.**
Fill in the template from section 4 with your actual numbers. 
What is missing? What do you still need to run?

**Questions to answer before you leave:**
1. What is your primary evaluation metric, and why is it the right choice?
2. Which component of your model contributes most to its performance?
3. How does your best result compare to the trivial baseline?
4. What ablations are still missing from your results table?